# Loadsmart — Analytics Engineer Challenge

Notebook with utility functions and exports from DuckDB.

**Prerequisites** (already in `.venv`):
```
pip install duckdb pandas paramiko
```

Run the pipeline before this notebook:
```bash
python scripts/ingest.py
cd dbt && dbt run
```

## 1. Setup

In [ ]:
import os
from pathlib import Path

import duckdb
import pandas as pd

PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == "notebooks" else Path(os.getcwd())
DUCKDB_PATH = str(PROJECT_ROOT / "data" / "loadsmart.duckdb")

print(f"DuckDB: {DUCKDB_PATH}")
print(f"File exists: {Path(DUCKDB_PATH).exists()}")

## 2. DuckDB connection

In [ ]:
con = duckdb.connect(DUCKDB_PATH, read_only=True)

# List available tables
con.execute("SHOW ALL TABLES").df()

## 3. `split_lane()` function

Takes a lane string in the form `"City,ST -> City,ST"` and returns a dict with origin and destination city and state.

In [ ]:
def split_lane(lane: str) -> dict:
   
    if not lane or " -> " not in lane:
        raise ValueError(f"Invalid lane format: {lane!r}. Expected: 'City,ST -> City,ST'")

    origin, destination = lane.split(" -> ", maxsplit=1)

    def _parse_side(side: str) -> tuple[str, str]:
        parts = side.split(",", maxsplit=1)
        if len(parts) != 2:
            raise ValueError(f"Malformed lane side: {side!r}")
        return parts[0].strip(), parts[1].strip()

    pickup_city, pickup_state = _parse_side(origin)
    delivery_city, delivery_state = _parse_side(destination)

    return {
        "pickup_city": pickup_city,
        "pickup_state": pickup_state,
        "delivery_city": delivery_city,
        "delivery_state": delivery_state,
    }


# --- Manual tests ---
cases = [
    ("Chicago,IL -> Dallas,TX",      {"pickup_city": "Chicago",   "pickup_state": "IL", "delivery_city": "Dallas",  "delivery_state": "TX"}),
    ("New York,NY -> Los Angeles,CA", {"pickup_city": "New York",  "pickup_state": "NY", "delivery_city": "Los Angeles", "delivery_state": "CA"}),
    (" Miami , FL ->  Atlanta , GA ", {"pickup_city": "Miami",     "pickup_state": "FL", "delivery_city": "Atlanta", "delivery_state": "GA"}),
]

for lane_str, expected in cases:
    result = split_lane(lane_str)
    assert result == expected, f"FAIL: {lane_str!r} → {result}"
    print(f"OK  {lane_str!r}")
    print(f"    {result}")

In [ ]:
# apply to a real sample from DuckDB
sample = con.execute("""
    SELECT lane_raw
    FROM main_mart.fct_shipments
    LIMIT 5
""").df()

pd.DataFrame(sample["lane_raw"].apply(split_lane).tolist())

In [ ]:
con.close()
print("DuckDB connection closed.")